In [1]:
import pandas as pd

data = pd.read_csv("spells_without_duplicates.csv")


In [2]:
data.columns


Index(['Name', 'Description', 'Range', 'Components', 'Material', 'Duration',
       'Casting.time', 'Level', 'School', 'Class', 'Ritual', 'Save', 'Damage',
       'Damage.type', 'Damage.progression'],
      dtype='str')

In [3]:
data


,Name,Description,Range,Components,Material,Duration,Casting.time,Level,School,Class,Ritual,Save,Damage,Damage.type,Damage.progression
0,Acid Arrow,A shimmering green arrow streaks toward a targ...,90 ft,V S M,Powdered rhubarb leaf and an adder's stomach,Instantaneous,1 action,2,Evocation,Wizard,NONE,NONE,4d4,Acid,NONE
1,Acid Splash,You hurl a bubble of acid. Choose one creature...,60 ft,V S,NONE,Instantaneous,1 action,0,Conjuration,"Sorcerer, Wizard",NONE,Dexterity,1d6,Acid,Cantrip Dice
2,Aid,Your spell bolsters your allies with toughness...,30 ft,V S M,A tiny strip of white cloth,8 hours,1 action,2,Abjuration,"Cleric, Paladin",NONE,NONE,NONE,NONE,NONE
3,Alarm,You set an alarm against unwanted intrusion. C...,30 ft,V S M,A tiny bell and a piece of fine silver wire,8 hours,1 minute,1,Abjuration,"Ranger, Wizard",Yes,NONE,NONE,NONE,NONE
4,Alter Self,You assume a different form. When you cast the...,Self,V S,NONE,Up to 1 hour,1 action,2,Transmutation,"Sorcerer, Wizard",NONE,NONE,NONE,NONE,NONE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,Wind Walk,You and up to ten willing creatures you can se...,30 ft,V S M,Fire and Holy Water,8 hours,1 minute,6,Transmutation,Druid,NONE,NONE,NONE,NONE,NONE
315,Wind Wall,A wall of strong wind rises from the ground at...,120 ft,V S M,A tiny fan and a feather of exotic origin,Up to 1 minute,1 action,3,Evocation,"Druid, Ranger",NONE,Strength,3d8,Bludgeoning,NONE
316,Wish,Wish is the mightiest spell a mortal creature ...,Self,V,NONE,Instantaneous,1 action,9,Conjuration,"Sorcerer, Wizard",NONE,NONE,NONE,NONE,NONE
317,Word of Recall,You and up to five willing creatures within 5 ...,5 ft,V,NONE,Instantaneous,1 action,6,Conjuration,Cleric,NONE,NONE,NONE,NONE,NONE


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

documents = [Document(page_content=str(description), metadata={'row_id': idx}) for idx, description in data['Description'].items()]

db = FAISS.from_documents(documents,
                          HuggingFaceEmbeddings(model_name='BAAI/bge-base-en-v1.5'))

In [9]:
def IR(query, k):
  retrieved_docs_direct = db.similarity_search(query, k=k)

  results = ""

  for doc in retrieved_docs_direct:
    row = doc.metadata.get('row_id')

    results += "Spell name: "
    results += str(data["Name"][row]) + "\n"
    results += 20*"-" + "\n"
    results += f"Class: {data['Class'][row]}\nLevel: {data['Level'][row]}\nSchool: {data['School'][row]}\nDuration: {data['Duration'][row]}\nCasting time: {data['Casting.time'][row]}\nComponents: {data['Components'][row]}\n"

    if data["Material"][row] != "NONE":
        results += f"Material: {data['Material'][row]}\n"

    if data["Ritual"][row] == "Yes":
        results += "Ritual\n"

    results += 20*"-" + "\n"
    results += "Description:\n"
    results += str(data["Description"][row]) + "\n\n"

  return results


In [10]:

query = "I want to stop time"
k = 4

print(IR(query, k))


# TODO:
#
# * Enable filtering before the query
# * Make it into Shiny


Spell name: Time Stop
--------------------
Class: Sorcerer, Wizard
Level: 9
School: Transmutation
Duration: Instantaneous
Casting time: 1 action
Components: V
--------------------
Description:
You briefly stop the flow of time for everyone but yourself. No time passes for other creatures, while you take 1d4+1 turns in a row, during which you can use actions and move as normal.This spell ends if one of the actions you use during this period, or any effects that you create during this period, affects a creature other than you or an object being worn or carried by someone other than you. In addition, the spell ends if you move to a place more than 1,000 feet from the location where you cast it.

Spell name: Arcane Lock
--------------------
Class: Wizard
Level: 2
School: Abjuration
Duration: Until dispelled
Casting time: 1 action
Components: V S M
Material: Gold dust worth at least 25gp, which the spell consumes
--------------------
Description:
You touch a closed door, window, gate, chest